In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)

from sklearn.model_selection import ParameterGrid

### 데이터 불러오기

In [2]:
# 파일 불러오기 
target = "Risk_Label"

# Load datasets
train_df = pd.read_csv(r"../../data/processed/ADASYN/data_train_adasyn.csv")
valid_df = pd.read_csv(r"../../data/processed/ADASYN/data_valid.csv")
test_df = pd.read_csv(r"../../data/processed/ADASYN/data_test.csv")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

train shape: (3270, 10) (3270,)
valid shape: (1438, 10) (1438,)
test shape: (822, 10) (822,)


## ParameterGrid SVM 적합

In [3]:
# =========================
# Parameter search space
# linear kernel에서는 gamma가 의미 없으므로 따로 분리
# =========================
param_grid = [
    {
        "svm__kernel": ["linear"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["rbf"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
    {
        "svm__kernel": ["poly"],
        "svm__C": [0.01, 0.1, 1, 10, 50],
        "svm__gamma": ["scale", "auto", 0.01, 0.1],
        "svm__class_weight": [None],
    },
]


def compute_metrics(y_true, y_pred):
    """
    Accuracy, F1, Recall, Precision, G-Mean 계산.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    g_mean = np.sqrt(sensitivity * specificity)

    return {
        "accuracy": acc,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "g_mean": g_mean,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "cm": cm,
    }


def make_threshold_candidates(scores):
    """
    SVM decision_function score 기준 threshold 후보 생성.
    score >= threshold 이면 high-risk(1)로 분류.

    validation score의 모든 unique 값을 후보로 사용해서
    G-Mean 최대 threshold를 더 정확하게 찾는다.
    """
    scores = np.asarray(scores, dtype=float)

    thresholds = np.unique(
        np.r_[
            np.unique(scores),
            0.0,
            scores.min() - 1e-12,
            scores.max() + 1e-12
        ]
    )

    return thresholds



GMEAN_WITHIN_RATIO = 0.99


def select_gmean99_precision(results):
    """
    선택 기준:
    1) validation G-Mean 최고값의 99% 이상인 후보만 유지
    2) 그 후보들 중 validation Precision 최대
    3) 동률이면 G-Mean → F1 → Recall → Specificity → Accuracy 순서
    """
    if not results:
        return None

    max_gmean = max(r["g_mean"] for r in results)
    eligible = [
        r for r in results
        if r["g_mean"] >= GMEAN_WITHIN_RATIO * max_gmean
    ]

    eligible = sorted(
        eligible,
        key=lambda r: (
            r["precision"],
            r["g_mean"],
            r["f1"],
            r["recall"],
            r["specificity"],
            r["accuracy"],
        ),
        reverse=True
    )

    return eligible[0]


### 지표

In [4]:
# =========================
# ParameterGrid + validation G-Mean 99% 이상 후보 중 Precision 우선 threshold search
# =========================
search_history = []
best_result = None

all_params = list(ParameterGrid(param_grid))

print(f"Total parameter combinations: {len(all_params)}")
print("선택 기준: validation G-Mean 최고값의 99% 이상 후보 중 Precision 최대")

for i, params in enumerate(all_params, start=1):

    model = Pipeline(
        steps=[("svm", SVC())]
    )
    model.set_params(**params)
    model.fit(X_train, y_train)

    valid_score_tmp = model.decision_function(X_valid)
    threshold_candidates = make_threshold_candidates(valid_score_tmp)

    threshold_results = []

    for threshold in threshold_candidates:
        valid_pred_tmp = (valid_score_tmp >= threshold).astype(int)
        metrics = compute_metrics(y_valid, valid_pred_tmp)

        result = {
            "params": params,
            "threshold": float(threshold),
            **{k: v for k, v in metrics.items() if k != "cm"},
        }
        threshold_results.append(result)

    # 각 파라미터 조합 내부에서 먼저 G-Mean 99% 이상 후보 중 Precision 최고 cutoff 선택
    best_for_params = select_gmean99_precision(threshold_results)
    search_history.append(best_for_params)

    print(
        f"[{i:>3}/{len(all_params)}] "
        f"G-Mean={best_for_params['g_mean']:.4f}, "
        f"Precision={best_for_params['precision']:.4f}, "
        f"F1={best_for_params['f1']:.4f}, "
        f"Recall={best_for_params['recall']:.4f}, "
        f"Threshold={best_for_params['threshold']:.6f}, "
        f"Params={params}"
    )

# 전체 파라미터 조합 중에서도 G-Mean 최고값의 99% 이상 후보만 남기고 Precision 최고 선택
best_result = select_gmean99_precision(search_history)

# =========================
# Best model 재학습
# =========================
best_params = best_result["params"]
best_threshold = best_result["threshold"]
best_valid_gmean = best_result["g_mean"]

best_svm = Pipeline(
    steps=[("svm", SVC())]
)
best_svm.set_params(**best_params)
best_svm.fit(X_train, y_train)

# Validation prediction using selected threshold
valid_score = best_svm.decision_function(X_valid)
valid_pred = (valid_score >= best_threshold).astype(int)

valid_metrics = compute_metrics(y_valid, valid_pred)
cm_valid = valid_metrics["cm"]

valid_acc = valid_metrics["accuracy"]
valid_f1 = valid_metrics["f1"]
valid_recall = valid_metrics["recall"]
valid_precision = valid_metrics["precision"]
valid_g_mean = valid_metrics["g_mean"]

print("\n" + "=" * 70)
print("Best SVM Model selected by validation G-Mean 99% + Precision")
print("=" * 70)
print("Best Params      :", best_params)
print("Best Threshold   :", best_threshold)
print("Best Valid G-Mean:", best_valid_gmean)
print("Best Valid Precision:", best_result["precision"])

print("\n[Validation] Metrics")
print(f"Accuracy  : {valid_acc:.4f}")
print(f"F1-Score  : {valid_f1:.4f}")
print(f"Recall    : {valid_recall:.4f}")
print(f"Precision : {valid_precision:.4f}")
print(f"G-Mean    : {valid_g_mean:.4f}")
print(f"\n[Validation] Confusion Matrix:\n{cm_valid}")

print("\nClassification Report:")
print(classification_report(y_valid, valid_pred, digits=4, zero_division=0))

# Search history 확인 및 저장
search_df_raw = pd.DataFrame(search_history)
max_gmean_all = search_df_raw["g_mean"].max()
search_df = search_df_raw[
    search_df_raw["g_mean"] >= GMEAN_WITHIN_RATIO * max_gmean_all
].copy()

search_df = search_df.sort_values(
    by=["precision", "g_mean", "f1", "recall", "specificity", "accuracy"],
    ascending=[False, False, False, False, False, False]
).reset_index(drop=True)

search_df.head()


Total parameter combinations: 45
선택 기준: validation G-Mean 최고값의 99% 이상 후보 중 Precision 최대
[  1/45] G-Mean=0.6510, Precision=0.2313, F1=0.3328, Recall=0.5934, Threshold=-0.443399, Params={'svm__C': 0.01, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  2/45] G-Mean=0.6682, Precision=0.2616, F1=0.3621, Recall=0.5879, Threshold=-0.066279, Params={'svm__C': 0.1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  3/45] G-Mean=0.6804, Precision=0.2429, F1=0.3550, Recall=0.6593, Threshold=-0.103494, Params={'svm__C': 1, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  4/45] G-Mean=0.6761, Precision=0.2437, F1=0.3535, Recall=0.6429, Threshold=-0.029072, Params={'svm__C': 10, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  5/45] G-Mean=0.6722, Precision=0.2484, F1=0.3548, Recall=0.6209, Threshold=0.029402, Params={'svm__C': 50, 'svm__class_weight': None, 'svm__kernel': 'linear'}
[  6/45] G-Mean=0.6455, Precision=0.2313, F1=0.3302, Recall=0.5769, Threshold=-0.184508, Params=

,params,threshold,accuracy,f1,recall,precision,sensitivity,specificity,g_mean,tn,fp,fn,tp
0,"{'svm__C': 1, 'svm__class_weight': None, 'svm_...",-0.103494,0.696801,0.355030,0.659341,0.242915,0.659341,0.702229,0.680447,882,374,62,120
1,"{'svm__C': 50, 'svm__class_weight': None, 'svm...",-0.096725,0.697497,0.353640,0.653846,0.242363,0.653846,0.703822,0.678374,884,372,63,119
2,"{'svm__C': 0.01, 'svm__class_weight': None, 's...",-0.067031,0.694715,0.353461,0.659341,0.241449,0.659341,0.699841,0.679289,879,377,62,120
3,"{'svm__C': 10, 'svm__class_weight': None, 'svm...",-0.118563,0.681502,0.353107,0.686813,0.237643,0.686813,0.680732,0.683766,855,401,57,125
4,"{'svm__C': 10, 'svm__class_weight': None, 'svm...",-0.118563,0.681502,0.353107,0.686813,0.237643,0.686813,0.680732,0.683766,855,401,57,125


### Test 성능 결과 

In [5]:
# =========================
# Test evaluation using selected SVM model and selected threshold
# =========================
test_score = best_svm.decision_function(X_test)
test_pred = (test_score >= best_threshold).astype(int)

test_metrics = compute_metrics(y_test, test_pred)
cm_test = test_metrics["cm"]

test_acc = test_metrics["accuracy"]
test_f1 = test_metrics["f1"]
test_recall = test_metrics["recall"]
test_precision = test_metrics["precision"]
test_g_mean = test_metrics["g_mean"]

print("\n" + "=" * 70)
print("[Test] Metrics")
print("=" * 70)
print("Selected Threshold:", best_threshold)
print(f"Accuracy  : {test_acc:.4f}")
print(f"F1-Score  : {test_f1:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"G-Mean    : {test_g_mean:.4f}")
print(f"\n[Test] Confusion Matrix:\n{cm_test}")

print("\nClassification Report:")
print(classification_report(y_test, test_pred, digits=4, zero_division=0))


[Test] Metrics
Selected Threshold: -0.10349434842088279
Accuracy  : 0.7105
F1-Score  : 0.3533
Recall    : 0.6566
Precision : 0.2416
G-Mean    : 0.6865

[Test] Confusion Matrix:
[[519 204]
 [ 34  65]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9385    0.7178    0.8135       723
           1     0.2416    0.6566    0.3533        99

    accuracy                         0.7105       822
   macro avg     0.5901    0.6872    0.5834       822
weighted avg     0.8546    0.7105    0.7581       822



In [6]:
print("Best validation G-Mean:", best_valid_gmean)
print("Best validation Precision:", best_result["precision"])
print("Selected threshold:", best_threshold)
print("Test G-Mean:", test_g_mean)

print("\nBest Params:")
print(best_params)

print("\nValid Confusion Matrix:")
print(confusion_matrix(y_valid, valid_pred, labels=[0, 1]))

print("\nTest Confusion Matrix:")
print(confusion_matrix(y_test, test_pred, labels=[0, 1]))

Best validation G-Mean: 0.6804471538998216
Best validation Precision: 0.242914979757085
Selected threshold: -0.10349434842088279
Test G-Mean: 0.6865206599504725

Best Params:
{'svm__C': 1, 'svm__class_weight': None, 'svm__kernel': 'linear'}

Valid Confusion Matrix:
[[882 374]
 [ 62 120]]

Test Confusion Matrix:
[[519 204]
 [ 34  65]]


In [7]:
# =========================
# Save predictions
#   Date + SVM_Pred만 저장
# =========================
result_dir = Path(r"../../results/results_ML")
result_dir.mkdir(parents=True, exist_ok=True)

# valid_df/test_df에 Date가 있으면 그대로 사용
if "Date" in valid_df.columns and "Date" in test_df.columns:
    valid_dates = pd.to_datetime(valid_df["Date"]).dt.strftime("%Y-%m-%d").reset_index(drop=True)
    test_dates = pd.to_datetime(test_df["Date"]).dt.strftime("%Y-%m-%d").reset_index(drop=True)

# Date가 없으면 data_selected.csv에서 chronological split 기준으로 가져옴
else:
    date_df = pd.read_csv(r"../../data/processed/data_selected.csv")
    date_df["Date"] = pd.to_datetime(date_df["Date"])
    date_df = date_df.sort_values("Date").reset_index(drop=True)

    n_valid = len(valid_pred)
    n_test = len(test_pred)

    valid_dates = (
        date_df["Date"]
        .tail(n_valid + n_test)
        .head(n_valid)
        .dt.strftime("%Y-%m-%d")
        .reset_index(drop=True)
    )

    test_dates = (
        date_df["Date"]
        .tail(n_test)
        .dt.strftime("%Y-%m-%d")
        .reset_index(drop=True)
    )

# Date + SVM_Pred만 저장
valid_pred_df = pd.DataFrame({
    "Date": valid_dates,
    "SVM_Pred": valid_pred
})

test_pred_df = pd.DataFrame({
    "Date": test_dates,
    "SVM_Pred": test_pred
})

valid_save_path = result_dir / "02. SVM_valid_pred.csv"
test_save_path = result_dir / "02. SVM_test_pred.csv"

valid_pred_df.to_csv(valid_save_path, index=False, encoding="utf-8-sig")
test_pred_df.to_csv(test_save_path, index=False, encoding="utf-8-sig")

print(f"\n✓ 저장 완료: {valid_save_path}")
print(f"✓ 저장 완료: {test_save_path}")


✓ 저장 완료: ..\..\results\results_ML\02. SVM_valid_pred.csv
✓ 저장 완료: ..\..\results\results_ML\02. SVM_test_pred.csv
